# Ch.1　データを「読む」── pandas で数字を掴む

| 項目 | 内容 |
|------|------|
| **使用データ** | ワインデータ（`load_wine()`）178件・13特徴量・3品種 |
| **作るもの** | 品種別・主要指標の集計表 |
| **実務での対応物** | 月次売上レポート / KPI ダッシュボードの集計表 |
| **演習時間** | 40分（**問3まで完了で合格**） |

---

### このChapterで身につけること

```
新しいデータをもらう
    ↓
① 形を確認する（head / info / describe / isnull）← 分析を始める前の必須ルーティン
    ↓
② 件数を確認する（value_counts）← グループのバランスを把握する
    ↓
③ グループ別に集計する（groupby）← 「差」を数字で見る
    ↓
④ 数字から意味を読む（考察）← データサイエンティストとしての本番
```

> ✅ **問3まで完了すれば十分です。問4は時間が余った人だけ取り組んでください。**

---
## 🤖 ブラウザ版AIの使い方

JupyterLab を左画面・AIを右画面に並べて作業してください。

| ツール | URL |
|--------|-----|
| Microsoft Copilot | https://copilot.microsoft.com |
| ChatGPT | https://chat.openai.com |

### AIを使うタイミング

| タイミング | ルール |
|-----------|--------|
| 座学中 | ❌ 禁止（まず概念を理解する） |
| 演習 問1〜2 | ✅ `ch1_prompt_guide.md` のテンプレートを使って質問する |
| 演習 問3（考察） | ✅ コードの質問はOK / **考察文は必ず自分で書く** |
| エラーが出たとき | ✅ すぐに使ってOK（エラーメッセージをそのまま貼る） |

> 📄 詳しいプロンプト集 → `ch1_prompt_guide.md`

---
## 🔧 準備 ── ライブラリの読み込み

**まずこのセルを実行してください。**

| ライブラリ | 役割 | 実務での対応 |
|-----------|------|------------|
| `pandas` | 表形式データの操作 | Excel の代わり。コードで再現・自動化できる |
| `numpy` | 数値計算の基盤 | 平均・パーセンタイルなどの計算に使う |
| `sklearn.datasets` | 練習用データの提供 | 今日だけ使う（実務では `pd.read_csv()` で読む） |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

print("✅ ライブラリの読み込み完了")

---
## データを読み込む

**実務での対応：**
```python
# 実務では CSV や Excel をこう読む
df = pd.read_csv("sales_data.csv")         # CSV の場合
df = pd.read_excel("monthly_report.xlsx")  # Excel の場合
```

今日は環境に依存しない内蔵データを使います。構造はまったく同じです。

In [ ]:
wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種"]  = wine.target
df["品種名"] = df["品種"].map({0: "Barolo", 1: "Grignolino", 2: "Barbera"})

print(f"✅ 読み込み完了  行数: {len(df)}  列数: {len(df.columns)}")

---
## 📋 データの「形」を確認する ── 分析前の必須ルーティン

> **なぜ最初に確認するのか：**  
> 「思っていたデータと違う」に気づかないまま進むと、後で全部やり直しになります。  
> プロのデータサイエンティストでも、新しいデータは必ずこの順番で確認します。

```
STEP 1: head()       → 「何が入っているか」を目で見る
STEP 2: info()       → 「型・欠損」を確認する
STEP 3: describe()   → 「値の大きさ・ばらつき」を把握する
STEP 4: isnull()     → 「空欄がないか」を数える
```

### STEP 1｜先頭5行を目で確認する（`head()`）

**見るポイント：**
- 列名が想定通りか（スペルミス・日本語混在など）
- 値の大きさは想定内か（桁数のズレ・単位の違いなど）
- 明らかにおかしい値（マイナスになってはいけない列が負など）はないか

In [ ]:
df.head()

**↑ 実行結果を見て確認してください：**
- `alcohol` 列：13〜14 付近の小数 → ワインのアルコール度数（%）として妥当
- `proline` 列：数百〜千台 → 他の列より桁が大きい（後で集計に影響する可能性あり）
- `品種名` 列：Barolo / Grignolino / Barbera の文字列 → 正常

### STEP 2｜型・件数・欠損値を確認する（`info()`）

**見るポイント：**
- `Non-Null Count` が行数（178）と一致していれば欠損なし
- `Dtype` が `object` になっている数値列は要注意（文字列として読み込まれた可能性）

In [ ]:
df.info()

**↑ 実行結果を見て確認してください：**
- `Non-Null Count` が全列 178 → 欠損なし ✅
- `Dtype` が `float64` / `int64` → 数値列として正常に読み込まれている ✅

> 💡 **実務でよくある問題：**  
> CSV を読み込んだとき `Dtype: object` になる数値列があれば、  
> その列に「文字列のような値」（例：`"N/A"` `"-"` `"1,000"`）が混入している可能性があります。

### STEP 3｜基本統計量を確認する（`describe()`）

**見るポイント：**
- `mean` と `max` の差が極端に大きい列 → 外れ値の可能性
- `std`（標準偏差）が大きい列 → グループ間の差が出やすい変数の候補

In [ ]:
df.describe().round(2)

**↑ 実行結果を見て注目してください：**

| 列名 | mean | max | std | 見るポイント |
|------|------|-----|-----|------------|
| `proline` | 746 | 1680 | 314 | std が大きく、max も突出している |
| `alcohol` | 13.0 | 14.8 | 0.81 | std は小さめ。品種間の差は「集計」で確認 |
| `color_intensity` | 5.1 | 13.0 | 2.32 | max が mean の 2.5 倍以上 → 外れ値かもしれない |

> 💡 **データサイエンティストのコツ：**  
> `describe()` の結果は「ストーリー」を読むように見ます。  
> 「`proline` は値のばらつきが大きいから、品種ごとに差があるかもしれない」のように仮説を立てながら読む。

### STEP 4｜欠損値を数える（`isnull().sum()`）

**実務での重要性：**  
欠損値は「見えないデータ」です。  
- グループ集計の平均が実態とズレる
- 機械学習モデルがエラーを出す  
- 「なぜ欠損しているか」自体がデータの意味を持つ場合もある

In [ ]:
missing = df.isnull().sum()
print(missing)
print(f"\n欠損値の合計: {missing.sum()} 件  ← 0 なら問題なし")

**↑ 実行結果：** 全列 0 → 今日のデータは欠損なし。安心して次に進めます。

> 💡 **実務で欠損が見つかったら：**
> ```
> 欠損が少ない（全体の5%以下）→ その行を削除（dropna()）
> 欠損が多い特定の列        → その列ごと削除（drop()）
> 欠損が多いが重要な列      → 平均値・中央値で補完（fillna()）
> ```
> どれを選ぶかは「なぜ欠損しているか」によって変わります。

---
## 問1｜品種ごとのデータ件数を確認する ★☆☆

### なぜ件数を最初に確認するのか

集計を始める前に「グループが均等かどうか」を確認するのが鉄則です。

**例えば、もし件数が偏っていたら…**
```
品種A: 170件
品種B:   5件  ← 少なすぎる
品種C:   3件  ← 少なすぎる
```
→ 品種B・Cの「平均値」は信頼性が低い（たった5件の平均）  
→ 機械学習モデルに学ばせると品種Aに偏った予測になる（不均衡データ問題）

**実務での場面：**
- 「各商品カテゴリが何件あるか」
- 「地域ごとの顧客数のバランスはどうか」
- 「正常データと異常データの比率はどのくらいか」

```python
# ヒント：このメソッドを使います
df["列名"].value_counts()
```

> 🤖 AIに聞く場合 → `ch1_prompt_guide.md` の **[P1-2]** を使ってください

In [ ]:
# 🎯 品種番号ごとの件数を数えてください
# ヒント: df["品種"].value_counts()



In [ ]:
# 🎯 品種名で表示してみましょう（番号より読みやすくなります）
# ヒント: df["品種名"].value_counts()



#### ✅ チェックポイント

**確認すること：**
- [ ] 3品種すべてが表示されているか
- [ ] 合計が **178 件** になっているか
- [ ] 最も多い品種と最も少ない品種の差は何倍か？

**結果の読み方：**

| 件数の差 | 判断 |
|---------|------|
| 2倍以内 | ほぼ均等。集計・分類に支障なし |
| 2〜5倍 | 注意が必要。分類モデルでは精度に差が出る可能性 |
| 5倍以上 | 不均衡データ。対処法を検討する必要あり |

> 💡 **今日のデータは均等に近い。次のChapter（Ch.3）で「このバランスがモデル精度に関係する」と気づけます。**

---
## 問2｜品種別の集計表を作る ★★☆

### `groupby()` の考え方

```
df.groupby("品種名")         ← 品種ごとに仕分ける
         ["alcohol"]        ← 対象列を選ぶ
         .mean()            ← 各グループの平均を計算する
```

**Excel のピボットテーブルと同じことを1行で書けます。**  
コードで書く利点：条件を変えて何度でも瞬時に再計算できる。

**実務での場面：**
- 「地域ごとの月次売上平均」
- 「年代別の購入単価と標準偏差」
- 「担当者別の成約率と案件数」

> 🤖 AIに聞く場合 → `ch1_prompt_guide.md` の **[P1-3]** を使ってください

In [ ]:
# 🎯 品種ごとのアルコール度数の平均を計算してください
# ヒント: df.groupby("品種名")["alcohol"].mean().round(2)



In [ ]:
# 🎯 品種ごとに「alcohol」と「proline」の平均をまとめて計算してください
# ヒント: df.groupby("品種名")[["列1", "列2"]].mean().round(2)



In [ ]:
# 🎯 さらに平均・最大・最小・標準偏差をまとめて出してください
# ヒント: .agg(["mean", "max", "min", "std"])



#### ✅ チェックポイント

**確認すること：**
- [ ] 3品種の数値が表示されているか
- [ ] Barolo のプロリン（proline）は他品種より明らかに大きいか（約2倍）
- [ ] std（標準偏差）が大きい品種はどれか

**結果の読み方（期待される傾向）：**

| 指標 | 注目ポイント | 意味 |
|------|-----------|------|
| `alcohol` の mean | 品種間で 1〜2 程度の差 | 差はあるが、1変数だけで品種を特定するのは難しい |
| `proline` の mean | Barolo だけ突出して高い | この変数だけで品種を「ある程度」識別できる |
| `std` | 大きい品種 = 値のばらつきが大きい | 「典型的な Barolo」より個体差が大きいということ |

> 💡 **ここで立てた仮説（「proline で品種を区別できそう」）は、  
> Ch.3 の特徴量重要度グラフで答え合わせができます。楽しみにしておいてください。**

---
## 問3｜集計結果から考察する ★★☆（最重要）

### なぜ考察が「最重要」なのか

> **数字を集計するのは手段、意味を読み取るのが目的です。**

データサイエンティストの仕事の大半は「コードを書くこと」ではなく、  
「数字を見て、ビジネスの言葉に翻訳して、次のアクションを提案すること」です。

集計表を作るだけなら Excel でもできます。  
「この数字から何が読み取れるか・何をすべきか」を言語化できる人が実務で価値を発揮します。

---

### 良い考察の構造（3ステップ）

```
① 事実（数字で言う）
   「Barolo の proline 平均は〇〇で、他品種の約△倍だった」

② 解釈（意味を言う）
   「→ proline はワイン品種を区別する特徴的な指標になりそう」

③ 次のアクション（どう活かすかを言う）
   「→ Ch.3 の分類モデルでは proline が重要な変数になると予測する」
```

> ✅ ここはAIに書いてもらわず、**自分の言葉で書いてください。**  
> 正解はありません。「気づいたこと」を素直に書くのが最重要です。

### 考察欄

---

**① 事実：数字で発見したこと**

> （例：「〇〇品種の △△ 平均は他品種の約〇倍だった」）

（ここに書いてください）

---

**② 解釈：その数字は何を意味すると思いますか？**

> 正解はありません。「なぜそうなるか」を自由に考えてください。

（ここに書いてください）

---

**③ 予測：Ch.3 の分類モデルで重要になりそうな変数はどれだと思いますか？**

> 理由も一言添えると◎

（ここに書いてください）

---
## 問4（発展）｜条件でデータを絞り込む ★★★

> **⏰ 時間が余った人だけ取り組んでください**

### 実務でのイメージ

「全体の平均」だけでは見えないことがあります。  
条件で絞り込んでから集計することで、特定の層の実態が見えてきます。

**実務での場面：**
```python
# 実務でよくある絞り込みの例（参考：実行不要）
df[df["売上"] >= 1000000]                        # 売上100万円以上の顧客だけ
df[(df["地域"] == "東京") & (df["年齢"] >= 30)]   # 東京の30歳以上だけ
df[df["購入日"] >= "2024-01-01"]                  # 2024年以降のデータだけ
```

> 🤖 AIに聞く場合 → `ch1_prompt_guide.md` の **[P1-4]** を使ってください

In [ ]:
# 🎯 アルコール度数が 13.5 以上のワインだけ取り出してください
# ヒント: df[df["alcohol"] >= 13.5]



In [ ]:
# 🎯 アルコール度数が 13.5 以上 かつ プロリンが 700 以上 の条件で絞り込んでください
# ⚠️ 複数条件のとき: df[(条件1) & (条件2)]  ← 括弧が必ず必要！



In [ ]:
# 🎯 絞り込んだデータで品種別の平均を計算してください
# （例：high_alc.groupby(...)）



#### ✅ チェックポイント

**確認すること：**
- [ ] 絞り込み後の件数は全体（178件）より少なくなっているか
- [ ] AND 条件で絞るほど、件数はさらに減っているか
- [ ] 絞り込み後の集計結果は、全体の集計と何が違うか

**結果の読み方：**

> 絞り込み後の集計が全体の集計と「似ている」 → 絞り込んだ条件は全体を代表している  
> 絞り込み後の集計が全体の集計と「かなり違う」 → 絞り込んだ層は特殊な特徴を持つ

> 💡 **実務でのポイント：**  
> 「高アルコール × 高プロリン」のワインは、どの品種に集中しますか？  
> この絞り込みが「ターゲットセグメントの分析」と同じ発想です。